# Clean Annotations

Cleans `data/chinese_traffic_signs/annotations.csv`:

- removes rows with no matching image file (**orphan rows**)
- removes repeated `file_name` rows, keeping the first (**duplicate rows**)

This is pure dataframe filtering - it never moves, renames or deletes any image file, and is safe to run every time (idempotent: a second run finds nothing left to clean).

The 84 assignment-test images (`../data/test-image-list.txt`) are **not** removed from the dataset here. They stay in `data/chinese_traffic_signs/` and are excluded from the training pool purely by filtering (`pipeline.annotations.split_held_out`), reused identically by every downstream stage - see the last cell below.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pipeline import annotations, paths

## 1. Read the raw annotation file and dataset images

In [2]:
annotations_file, raw_annotations_df = annotations.read_raw_annotations()
image_lookup = {p.name: p for p in annotations.list_dataset_images()}

print("Annotation file:", annotations_file.resolve())
print("Raw annotation rows:", len(raw_annotations_df))
print("Dataset images on disk:", len(image_lookup))

Annotation file: /home/hongye/projects/traffic-sign-recognition/data/chinese_traffic_signs/annotations.csv
Raw annotation rows: 5998
Dataset images on disk: 5998


## 2. Find orphan and duplicate rows

In [3]:
cleaned_df, orphan_count, duplicate_count = annotations.clean_annotations(
    raw_annotations_df, image_lookup
)

print("Orphan rows removed (no matching image):", orphan_count)
print("Duplicate file_name rows removed:", duplicate_count)
print("Rows before cleanup:", len(raw_annotations_df))
print("Rows after cleanup:", len(cleaned_df))
print("Classes after cleanup:", cleaned_df["category"].nunique())

Orphan rows removed (no matching image): 0
Duplicate file_name rows removed: 0
Rows before cleanup: 5998
Rows after cleanup: 5998
Classes after cleanup: 58


## 3. Write the cleaned annotations back

Overwrites `annotations.csv` in place with the cleaned table. Safe to rerun: a second run reads back the already-clean file and finds 0 rows to remove.

In [4]:
cleaned_df.to_csv(annotations_file, index=False)

print("Wrote cleaned annotations to:", annotations_file.resolve())

Wrote cleaned annotations to: /home/hongye/projects/traffic-sign-recognition/data/chinese_traffic_signs/annotations.csv


## 4. Held-out assignment-test filenames

The 84 filenames in `test-image-list.txt` are excluded from the training pool by filtering, not by moving files. `pipeline.annotations.split_held_out` is the single place this split happens - every downstream notebook (feature extraction, classifiers) calls it the same way.

In [5]:
held_out = annotations.held_out_filenames()
train_pool_df, held_out_df = annotations.split_held_out(cleaned_df, held_out)

print("Held-out filenames in manifest:", len(held_out))
print("Held-out rows found in annotations:", len(held_out_df))
print("Training-pool rows:", len(train_pool_df))

annotations.assert_no_leakage(train_pool_df["file_name"], held_out_df["file_name"])
print("No leakage between the training pool and the held-out set.")

Held-out filenames in manifest: 84
Held-out rows found in annotations: 84
Training-pool rows: 5914
No leakage between the training pool and the held-out set.


## Note

Earlier versions of this notebook physically moved the 84 held-out images into a timestamped `data/removed_input_duplicates/<timestamp>/` backup folder, gated behind a manual `ACTION`/confirmation-phrase toggle before every run. That's gone: **no image file is ever moved, renamed or deleted**. The held-out filenames stay in `data/chinese_traffic_signs/` permanently and are excluded only when building the training pool, via `pipeline.annotations.split_held_out` above - which is why there is nothing to "restore" and nothing for downstream notebooks to search a backup folder for.